# 🧠 Brain Tumor Segmentation — U-Net + ResNet50

**Architecture:** U-Net decoder + ResNet50 encoder (pretrained ImageNet)  
**Dataset:** LGG MRI Segmentation — https://www.kaggle.com/datasets/mateuszbuda/lgg-mri-segmentation  

### U-Net + ResNet50
| Yếu tố | Chi tiết |
|---|---|
| Encoder mạnh | ResNet50 pretrained ImageNet — học features phong phú ngay từ đầu |
| Skip connections | U-Net giữ spatial detail → biên khối u sắc nét |
| Phù hợp dataset vừa | ResNet50 ít overfit hơn EfficientNet-B4 trên ~1,500 ảnh |
| Tốc độ | Nhanh hơn ~30% so với EfficientNet-B4 |


git clone https://github.com/NguyenTienDung200705/BrainSeg-XAI.git
cd " vào project nhé"
git checkout -b feature/unet-resnet50
git add notebooks/training1.ipynb  
git commit -m "Add UNet ResNet50 training notebook"
git push -u origin feature/unet-resnet50

In [ ]:
# Cài đặt — bỏ comment nếu cần
# !pip install segmentation-models-pytorch timm albumentations -q
# !pip install torch torchvision opencv-python-headless scikit-image scipy tqdm -q

# Download dataset từ Kaggle:
# !kaggle datasets download -d mateuszbuda/lgg-mri-segmentation -q
# !unzip -q lgg-mri-segmentation.zip -d dataset/


In [ ]:
import os, sys, random, glob, json, warnings
warnings.filterwarnings('ignore')

import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast
import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2
from tqdm import tqdm
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path


DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')

In [ ]:
#  CONFIG
CFG = {
    # Model 
    'arch'            : 'unet',           # U-Net decoder
    'encoder'         : 'resnet50',       # ResNet50 encoder pretrained ImageNet
    'encoder_weights' : 'imagenet',

    # Data 
    'data_root'       : '/content/dataset/kaggle_3m',
    'img_size'        : 384,              # 384×384 — giữ detail tốt hơn 256
    'val_split'       : 0.15,
    'num_workers'     : 4,
    'no_tumor_ratio'  : 0.1,             # giữ 10% no-tumor để model không bị bias

    # Training
    'epochs'          : 80,
    'batch_size'      : 8,               # giảm xuống 4 nếu OOM với 384
    'accumulate_grad' : 2,               # gradient accumulation (effective batch = 8×2=16)
    'lr'              : 3e-4,            # decoder learning rate
    'encoder_lr_ratio': 0.1,             # encoder LR = lr × ratio (pretrained → cần nhỏ hơn)
    'weight_decay'    : 1e-4,
    'warmup_epochs'   : 5,
    'amp'             : True,            # mixed precision FP16

    # Loss weights
    'w_dice'          : 0.5,
    'w_tversky'       : 0.3,
    'w_bce'           : 0.2,
    'pos_weight'      : 12.0,            # BCELoss weight cho tumor pixels
    'tversky_alpha'   : 0.7,             # penalize False Negative nhiều hơn
    'tversky_beta'    : 0.3,
    'tversky_gamma'   : 0.75,

    # Regularization
    'decoder_dropout' : 0.2,             # dropout trong 2 decoder blocks cuối

    # LR schedule
    'plateau_patience': 7,               # ReduceLROnPlateau patience
    'plateau_factor'  : 0.5,

    # Early stopping
    'early_stop'      : 15,

    # Save
    'save_dir'        : '/content/dataset',
    'save_name'       : 'unet_best.pth',
}

os.makedirs(CFG['save_dir'], exist_ok=True)
os.makedirs('../outputs', exist_ok=True)

print('CONFIG')
print(f"  Model     : {CFG['arch'].upper()} + {CFG['encoder']}")
print(f"  Img size  : {CFG['img_size']}×{CFG['img_size']}")
print(f"  Batch     : {CFG['batch_size']} × accum {CFG['accumulate_grad']} = {CFG['batch_size']*CFG['accumulate_grad']} effective")
print(f"  Epochs    : {CFG['epochs']} (early stop patience={CFG['early_stop']})")
print(f"  LR        : decoder={CFG['lr']:.0e}  encoder={CFG['lr']*CFG['encoder_lr_ratio']:.0e}")
print(f"  pos_weight: {CFG['pos_weight']} | dropout: {CFG['decoder_dropout']}")


In [ ]:
#  DATASET & AUGMENTATION

def get_transforms(mode: str, size: int = 384):
    """
    train: heavy augmentation để tránh overfit
    val  : chỉ resize + normalize
    tta  : 1 flip để dùng trong Test-Time Augmentation
    """
    mean = [0.485, 0.456, 0.406]
    std  = [0.229, 0.224, 0.225]

    if mode == 'train':
        return A.Compose([
            A.Resize(size, size),

            # Geometric
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.3),
            A.RandomRotate90(p=0.5),
            A.ShiftScaleRotate(
                shift_limit=0.1, scale_limit=0.15, rotate_limit=25,
                border_mode=cv2.BORDER_REFLECT_101, p=0.6
            ),
            A.ElasticTransform(
                alpha=80, sigma=80 * 0.05, alpha_affine=80 * 0.03,
                border_mode=cv2.BORDER_REFLECT_101, p=0.25
            ),
            A.GridDistortion(num_steps=5, distort_limit=0.2, p=0.2),

            # Intensity (chỉ ảnh, không áp lên mask)
            A.OneOf([
                A.RandomBrightnessContrast(
                    brightness_limit=0.25, contrast_limit=0.25, p=1.0
                ),
                A.RandomGamma(gamma_limit=(75, 125), p=1.0),
                A.CLAHE(clip_limit=3.0, tile_grid_size=(8, 8), p=1.0),
            ], p=0.6),
            A.OneOf([
                A.GaussianBlur(blur_limit=(3, 5), p=1.0),
                A.MedianBlur(blur_limit=3, p=1.0),
            ], p=0.2),
            A.GaussNoise(var_limit=(5.0, 30.0), p=0.2),

            # Dropout / cutout
            A.CoarseDropout(
                max_holes=6, max_height=24, max_width=24,
                min_holes=1, fill_value=0, p=0.25
            ),

            A.Normalize(mean=mean, std=std),
            ToTensorV2(),
        ])

    else:  # val hoặc test
        return A.Compose([
            A.Resize(size, size),
            A.Normalize(mean=mean, std=std),
            ToTensorV2(),
        ])


class BrainMRIDataset(Dataset):
    def __init__(self, img_paths, mask_paths, transform=None):
        self.imgs      = img_paths
        self.masks     = mask_paths
        self.transform = transform

    def __len__(self):
        return len(self.imgs)

    def __getitem__(self, idx):
        img  = cv2.imread(self.imgs[idx])
        img  = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        mask = cv2.imread(self.masks[idx], cv2.IMREAD_GRAYSCALE)
        mask = (mask > 127).astype(np.uint8)

        if self.transform:
            aug  = self.transform(image=img, mask=mask)
            img  = aug['image']                         # Tensor C×H×W
            mask = aug['mask'].float().unsqueeze(0)     # Tensor 1×H×W
        return img, mask


# Scan dataset 
all_imgs  = sorted(glob.glob(
    os.path.join(CFG['data_root'], '**', '*[!_mask].tif'), recursive=True
))
all_masks = sorted(glob.glob(
    os.path.join(CFG['data_root'], '**', '*_mask.tif'), recursive=True
))
assert len(all_imgs) > 0, f"Không tìm thấy ảnh tại {CFG['data_root']}"
assert len(all_imgs) == len(all_masks), "Số ảnh và mask không khớp"
print(f'Tổng ảnh: {len(all_imgs)} | Tổng mask: {len(all_masks)}')

# Phân loại có / không có khối u 
pairs = list(zip(all_imgs, all_masks))
random.seed(42)
random.shuffle(pairs)

has_tumor, no_tumor = [], []
print('Scanning masks...', end='', flush=True)
for i, (ip, mp) in enumerate(pairs):
    m = cv2.imread(mp, cv2.IMREAD_GRAYSCALE)
    if m is None:
        continue
    (has_tumor if m.max() > 0 else no_tumor).append((ip, mp))
    if (i + 1) % 200 == 0:
        print('.', end='', flush=True)
print(f' done')
print(f'  Có khối u : {len(has_tumor)}')
print(f'  Không có  : {len(no_tumor)}')

# Cân bằng dataset 
keep_no  = int(len(has_tumor) * CFG['no_tumor_ratio'])
balanced = has_tumor + no_tumor[:keep_no]
random.shuffle(balanced)
print(f'  Sử dụng  : {len(balanced)} ảnh '
      f'(has_tumor={len(has_tumor)}, no_tumor={keep_no})')

# Train / Val split
n_val       = max(1, int(len(balanced) * CFG['val_split']))
val_pairs   = balanced[:n_val]
train_pairs = balanced[n_val:]

train_imgs, train_masks = zip(*train_pairs)
val_imgs,   val_masks   = zip(*val_pairs)

train_ds = BrainMRIDataset(
    list(train_imgs), list(train_masks),
    transform=get_transforms('train', CFG['img_size'])
)
val_ds = BrainMRIDataset(
    list(val_imgs), list(val_masks),
    transform=get_transforms('val', CFG['img_size'])
)

train_loader = DataLoader(
    train_ds, batch_size=CFG['batch_size'], shuffle=True,
    num_workers=CFG['num_workers'], pin_memory=True, drop_last=True
)
val_loader = DataLoader(
    val_ds, batch_size=CFG['batch_size'], shuffle=False,
    num_workers=CFG['num_workers'], pin_memory=True
)
print(f'Train: {len(train_ds)} ảnh — {len(train_loader)} batches')
print(f'Val  : {len(val_ds)} ảnh  — {len(val_loader)} batches')


In [ ]:
#  LOSS FUNCTIONS
#  Dice + Focal-Tversky + BCE(pos_weight)

POS_WEIGHT = torch.tensor([CFG['pos_weight']], device=DEVICE)


def dice_loss(pred: torch.Tensor, target: torch.Tensor, smooth: float = 1.0):
    """Soft Dice Loss — phạt cả FP và FN đều nhau"""
    pred   = torch.sigmoid(pred).contiguous().view(-1)
    target = target.contiguous().view(-1)
    inter  = (pred * target).sum()
    return 1.0 - (2.0 * inter + smooth) / (pred.sum() + target.sum() + smooth)


def focal_tversky_loss(
    pred   : torch.Tensor,
    target : torch.Tensor,
    alpha  : float = CFG['tversky_alpha'],   # weight FN (bỏ sót khối u)
    beta   : float = CFG['tversky_beta'],    # weight FP
    gamma  : float = CFG['tversky_gamma'],   # focal exponent
    smooth : float = 1.0,
):
    """
    Focal Tversky Loss — penalize False Negative nặng hơn FP.
    Rất hiệu quả với imbalanced medical segmentation.
    alpha > beta → ưu tiên recall (không bỏ sót khối u)
    """
    pred   = torch.sigmoid(pred).contiguous().view(-1)
    target = target.contiguous().view(-1)
    TP = (pred * target).sum()
    FP = ((1.0 - target) * pred).sum()
    FN = (target * (1.0 - pred)).sum()
    tversky_index = (TP + smooth) / (TP + alpha * FN + beta * FP + smooth)
    return (1.0 - tversky_index) ** gamma


def bce_loss(pred: torch.Tensor, target: torch.Tensor):
    """BCE với pos_weight để xử lý class imbalance"""
    return F.binary_cross_entropy_with_logits(pred, target, pos_weight=POS_WEIGHT)


def combined_loss(pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    return (
        CFG['w_dice']    * dice_loss(pred, target) +
        CFG['w_tversky'] * focal_tversky_loss(pred, target) +
        CFG['w_bce']     * bce_loss(pred, target)
    )


# ── Metrics ──────────────────────────────────────────────────
def dice_score(pred, target, thresh=0.5, smooth=1.0):
    pred   = (torch.sigmoid(pred) > thresh).float().contiguous().view(-1)
    target = target.contiguous().view(-1)
    inter  = (pred * target).sum()
    return ((2.0 * inter + smooth) / (pred.sum() + target.sum() + smooth)).item()


def iou_score(pred, target, thresh=0.5, smooth=1.0):
    pred   = (torch.sigmoid(pred) > thresh).float().contiguous().view(-1)
    target = target.contiguous().view(-1)
    inter  = (pred * target).sum()
    union  = pred.sum() + target.sum() - inter
    return ((inter + smooth) / (union + smooth)).item()


def precision_recall(pred, target, thresh=0.5, smooth=1.0):
    pred   = (torch.sigmoid(pred) > thresh).float().contiguous().view(-1)
    target = target.contiguous().view(-1)
    TP = (pred * target).sum()
    prec = ((TP + smooth) / (pred.sum() + smooth)).item()
    rec  = ((TP + smooth) / (target.sum() + smooth)).item()
    return prec, rec


print('Loss functions defined ✓')
print(f'  Dice       weight : {CFG["w_dice"]}')
print(f'  FocalTversky weight: {CFG["w_tversky"]}  (alpha={CFG["tversky_alpha"]}, beta={CFG["tversky_beta"]})')
print(f'  BCE        weight : {CFG["w_bce"]}  (pos_weight={CFG["pos_weight"]})')


In [ ]:
#  MODEL: U-Net + ResNet50 (segmentation-models-pytorch)

model = smp.Unet(
    encoder_name     = CFG['encoder'],          # resnet50
    encoder_weights  = CFG['encoder_weights'],  # imagenet
    in_channels      = 3,
    classes          = 1,
    activation       = None,                    # raw logits — sigmoid trong loss
    decoder_use_batchnorm = True,
)

# ── Inject Dropout vào 2 decoder blocks cuối ────────────────
# Tránh overfit: ResNet50 mạnh nhưng dataset chỉ ~1,500 ảnh
if CFG['decoder_dropout'] > 0:
    _drop2d = nn.Dropout2d(p=CFG['decoder_dropout'])

    def _make_drop_hook(drop_layer):
        def hook(module, inp, output):
            if module.training:
                return drop_layer(output)
        return hook

    decoder_blocks = list(model.decoder.blocks)
    n_inject = min(2, len(decoder_blocks))
    for blk in decoder_blocks[-n_inject:]:
        # Hook sau conv2 của mỗi block
        if hasattr(blk, 'conv2'):
            blk.conv2.register_forward_hook(_make_drop_hook(_drop2d))
    print(f'Dropout(p={CFG["decoder_dropout"]}) injected vào {n_inject} decoder blocks cuối')

model = model.to(DEVICE)

# Thống kê model
total_params    = sum(p.numel() for p in model.parameters())
encoder_params  = sum(p.numel() for p in model.encoder.parameters())
decoder_params  = total_params - encoder_params

print(f'\nModel: {CFG["arch"].upper()} + {CFG["encoder"]}')
print(f'  Total params   : {total_params / 1e6:.1f}M')
print(f'  Encoder params : {encoder_params / 1e6:.1f}M  (pretrained={CFG["encoder_weights"]})')
print(f'  Decoder params : {decoder_params / 1e6:.1f}M  (random init)')
print(f'  Input size     : 3×{CFG["img_size"]}×{CFG["img_size"]}')

# Test forward pass
_dummy = torch.randn(1, 3, CFG['img_size'], CFG['img_size']).to(DEVICE)
with torch.no_grad():
    _out = model(_dummy)
print(f'  Output shape   : {tuple(_out.shape)}  ✓')
del _dummy, _out


In [ ]:
#  OPTIMIZER + LR SCHEDULER

# Encoder LR nhỏ hơn decoder vì đã pretrained
_enc_params = list(model.encoder.parameters())
_dec_params = [p for n, p in model.named_parameters()
               if not n.startswith('encoder')]

optimizer = torch.optim.AdamW(
    [
        {'params': _enc_params, 'lr': CFG['lr'] * CFG['encoder_lr_ratio'], 'name': 'encoder'},
        {'params': _dec_params, 'lr': CFG['lr'],                           'name': 'decoder'},
    ],
    weight_decay = CFG['weight_decay'],
)

# Warmup + Cosine Annealing
# Epoch 0→warmup : LR tăng tuyến tính từ 0 → lr
# Epoch warmup→end: LR giảm cosine đến 5% lr (không về 0)
def _warmup_cosine_schedule(epoch):
    w = CFG['warmup_epochs']
    T = CFG['epochs']
    if epoch < w:
        return (epoch + 1) / w
    ratio = (epoch - w) / max(T - w, 1)
    return max(0.05, 0.5 * (1.0 + np.cos(np.pi * ratio)))

scheduler_cosine = torch.optim.lr_scheduler.LambdaLR(
    optimizer, lr_lambda=_warmup_cosine_schedule
)

# ReduceLROnPlateau
# Nếu val_dice không cải thiện sau 7 epoch → giảm LR ×0.5
scheduler_plateau = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode      = 'max',
    factor    = CFG['plateau_factor'],
    patience  = CFG['plateau_patience'],
    min_lr    = 5e-8
)

# AMP scaler cho mixed precision
USE_AMP = CFG['amp'] and torch.cuda.is_available()
scaler  = GradScaler(enabled=USE_AMP)

print('Optimizer  : AdamW')
print(f'  Encoder LR : {CFG["lr"] * CFG["encoder_lr_ratio"]:.0e}')
print(f'  Decoder LR : {CFG["lr"]:.0e}')
print(f'Scheduler  : Warmup({CFG["warmup_epochs"]} ep) + CosineAnnealing + ReduceLROnPlateau')
print(f'AMP        : {USE_AMP}')
print(f'Grad Accum : {CFG["accumulate_grad"]} steps (effective batch = {CFG["batch_size"]*CFG["accumulate_grad"]})')


In [ ]:
#  TRAINING LOOP

best_dice  = 0.0
best_epoch = 0
no_improve = 0
history    = {
    'train_loss': [], 'val_loss': [],
    'val_dice'  : [], 'val_iou' : [],
    'val_prec'  : [], 'val_rec' : [],
    'lr'        : [],
}

print(f'\nBắt đầu training — {CFG["epochs"]} epochs')
print(f'Early stop patience: {CFG["early_stop"]} epochs\n')

for epoch in range(1, CFG['epochs'] + 1):

    # TRAIN
    model.train()
    train_loss = 0.0
    optimizer.zero_grad()

    bar = tqdm(
        enumerate(train_loader), total=len(train_loader),
        desc=f'Ep {epoch:3d}/{CFG["epochs"]} [Train]', ncols=100, leave=False
    )
    for step, (imgs, masks) in bar:
        imgs  = imgs.to(DEVICE, non_blocking=True)
        masks = masks.to(DEVICE, non_blocking=True)

        with autocast(enabled=USE_AMP):
            preds = model(imgs)
            loss  = combined_loss(preds, masks)
            loss  = loss / CFG['accumulate_grad']   # normalize cho grad accum

        scaler.scale(loss).backward()

        if (step + 1) % CFG['accumulate_grad'] == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        train_loss += loss.item() * CFG['accumulate_grad']
        bar.set_postfix(loss=f'{loss.item() * CFG["accumulate_grad"]:.4f}')

    # Flush remaining gradients nếu dataset không chia hết cho accumulate_grad
    if len(train_loader) % CFG['accumulate_grad'] != 0:
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad()

    train_loss /= len(train_loader)

    # VALIDATE
    model.eval()
    val_loss = val_dice = val_iou = val_prec = val_rec = 0.0

    with torch.no_grad():
        for imgs, masks in val_loader:
            imgs  = imgs.to(DEVICE, non_blocking=True)
            masks = masks.to(DEVICE, non_blocking=True)
            preds = model(imgs)

            val_loss += combined_loss(preds, masks).item()
            val_dice += dice_score(preds, masks)
            val_iou  += iou_score(preds, masks)
            p, r     = precision_recall(preds, masks)
            val_prec += p
            val_rec  += r

    n = len(val_loader)
    val_loss /= n; val_dice /= n; val_iou /= n
    val_prec /= n; val_rec  /= n

    # SCHEDULERS
    scheduler_cosine.step()
    scheduler_plateau.step(val_dice)

    lr_now = optimizer.param_groups[1]['lr']   # decoder LR
    gap    = val_loss - train_loss

    # LOG
    for k, v in [('train_loss', train_loss), ('val_loss', val_loss),
                  ('val_dice', val_dice), ('val_iou', val_iou),
                  ('val_prec', val_prec), ('val_rec', val_rec), ('lr', lr_now)]:
        history[k].append(v)

    flag = ''
    if val_dice > best_dice:
        best_dice  = val_dice
        best_epoch = epoch
        no_improve = 0
        torch.save({
            'epoch'           : epoch,
            'arch'            : CFG['arch'],
            'encoder'         : CFG['encoder'],
            'model_state_dict': model.state_dict(),
            'optimizer_state' : optimizer.state_dict(),
            'best_dice'       : best_dice,
            'cfg'             : CFG,
        }, f'{CFG["save_dir"]}/{CFG["save_name"]}')
        flag = '  ← best ✓'
    else:
        no_improve += 1

    print(
        f'Ep {epoch:3d} | '
        f'Loss {train_loss:.4f} → {val_loss:.4f} (Δ{gap:+.4f}) | '
        f'Dice {val_dice:.4f} | IoU {val_iou:.4f} | '
        f'P {val_prec:.3f} R {val_rec:.3f} | '
        f'LR {lr_now:.2e}{flag}'
    )

    # EARLY STOPPING
    if no_improve >= CFG['early_stop']:
        print(f'\n⏹  Early stopping tại epoch {epoch} '
              f'(best Dice={best_dice:.4f} ở epoch {best_epoch})')
        break

print(f'\n✅ Training xong!  Best Val Dice = {best_dice:.4f}  (epoch {best_epoch})')
print(f'   Weights lưu tại : {CFG["save_dir"]}/{CFG["save_name"]}')


In [ ]:
#  TÌM OPTIMAL THRESHOLD TRÊN VAL SET
#  Threshold mặc định 0.5 không phải lúc nào cũng tốt nhất

# Load best weights
print('Loading best weights...')
ckpt = torch.load(f'{CFG["save_dir"]}/{CFG["save_name"]}', map_location=DEVICE)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print(f'Loaded epoch {ckpt["epoch"]}  (Dice={ckpt["best_dice"]:.4f})')

# Collect tất cả predictions
all_probs, all_targets = [], []
with torch.no_grad():
    for imgs, masks in tqdm(val_loader, desc='Collecting predictions'):
        imgs = imgs.to(DEVICE)
        prob = torch.sigmoid(model(imgs)).cpu().numpy()
        all_probs.append(prob)
        all_targets.append(masks.numpy())

all_probs   = np.concatenate(all_probs,   axis=0)   # N×1×H×W
all_targets = np.concatenate(all_targets, axis=0)   # N×1×H×W

# Grid search threshold
thresholds  = np.arange(0.25, 0.75, 0.02)
dice_scores = []
for t in thresholds:
    pred_bin = (all_probs > t).astype(np.float32)
    inter    = (pred_bin * all_targets).sum()
    d        = (2.0 * inter + 1.0) / (pred_bin.sum() + all_targets.sum() + 1.0)
    dice_scores.append(d)

best_thresh      = float(thresholds[np.argmax(dice_scores)])
best_thresh_dice = float(max(dice_scores))

print(f'\nOptimal threshold : {best_thresh:.3f}')
print(f'Dice @ default 0.5: {dice_scores[np.argmin(np.abs(thresholds - 0.5))]:.4f}')
print(f'Dice @ optimal    : {best_thresh_dice:.4f}  (+{best_thresh_dice - dice_scores[np.argmin(np.abs(thresholds - 0.5))]:.4f})')

# Plot
fig, ax = plt.subplots(figsize=(9, 4), facecolor='#060d14')
ax.set_facecolor('#0a1520')
ax.plot(thresholds, dice_scores, color='#00c8ff', linewidth=2, marker='o', markersize=4)
ax.axvline(best_thresh, color='#00ff88', linestyle='--', linewidth=1.5,
           label=f'Best t={best_thresh:.2f}  Dice={best_thresh_dice:.4f}')
ax.axvline(0.5, color='#ff6b35', linestyle=':', linewidth=1.5, label='Default t=0.50')
ax.set_xlabel('Threshold', color='#aaa'); ax.set_ylabel('Dice Score', color='#aaa')
ax.set_title('Threshold Search', color='#e8f4f8')
ax.legend(facecolor='#060d14', labelcolor='#ccc')
ax.tick_params(colors='#888')
for spine in ax.spines.values(): spine.set_color('#1a2a3a')
plt.tight_layout()
plt.savefig('../outputs/threshold_search.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → ../outputs/threshold_search.png')


In [ ]:
#  TEST-TIME AUGMENTATION (TTA) — trung bình 8 predictions

def tta_predict(model: nn.Module, img_tensor: torch.Tensor) -> torch.Tensor:
    """
    img_tensor : B×C×H×W
    return     : B×1×H×W (averaged probability)
    """
    augment   = [
        lambda x: x,                                              # original
        lambda x: torch.flip(x, [3]),                            # flip H
        lambda x: torch.flip(x, [2]),                            # flip V
        lambda x: torch.rot90(x, 1, [2, 3]),                     # rot90
        lambda x: torch.rot90(x, 2, [2, 3]),                     # rot180
        lambda x: torch.rot90(x, 3, [2, 3]),                     # rot270
        lambda x: torch.flip(torch.rot90(x, 1, [2, 3]), [3]),    # rot90 + flipH
        lambda x: torch.flip(torch.rot90(x, 3, [2, 3]), [3]),    # rot270 + flipH
    ]
    deaugment = [
        lambda x: x,
        lambda x: torch.flip(x, [3]),
        lambda x: torch.flip(x, [2]),
        lambda x: torch.rot90(x, 3, [2, 3]),
        lambda x: torch.rot90(x, 2, [2, 3]),
        lambda x: torch.rot90(x, 1, [2, 3]),
        lambda x: torch.flip(torch.rot90(x, 3, [2, 3]), [3]),
        lambda x: torch.flip(torch.rot90(x, 1, [2, 3]), [3]),
    ]
    model.eval()
    preds = []
    with torch.no_grad():
        for aug, deaug in zip(augment, deaugment):
            x_aug = aug(img_tensor)
            prob  = torch.sigmoid(model(x_aug))
            preds.append(deaug(prob))
    return torch.stack(preds, dim=0).mean(dim=0)   # B×1×H×W


# Đánh giá TTA trên val set
print('Evaluating TTA on val set...')
tta_dice_list = []

for imgs, masks in tqdm(val_loader, desc='TTA Eval'):
    imgs  = imgs.to(DEVICE)
    probs = tta_predict(model, imgs).cpu()
    preds = (probs > best_thresh).float()

    inter = (preds * masks).sum()
    d     = (2.0 * inter + 1.0) / (preds.sum() + masks.sum() + 1.0)
    tta_dice_list.append(d.item())

tta_dice_mean = np.mean(tta_dice_list)
print(f'\nDice @ best_thresh (no TTA) : {best_thresh_dice:.4f}')
print(f'Dice @ best_thresh (TTA×8)  : {tta_dice_mean:.4f}  '
      f'(+{tta_dice_mean - best_thresh_dice:.4f})')


In [ ]:
#  VISUALIZE — xem kết quả trên 4 val samples

MEAN = np.array([0.485, 0.456, 0.406])
STD  = np.array([0.229, 0.224, 0.225])

def denormalize(tensor):
    img = tensor.permute(1, 2, 0).numpy()
    img = (img * STD + MEAN).clip(0, 1)
    return (img * 255).astype(np.uint8)


model.eval()
val_iter       = iter(val_loader)
imgs_b, masks_b = next(val_iter)

# Lấy batch có khối u để visualize rõ hơn
for imgs_b, masks_b in val_loader:
    if masks_b.sum() > 100:   # batch có đủ tumor pixels
        break

with torch.no_grad():
    probs_b = tta_predict(model, imgs_b.to(DEVICE)).cpu()

n_show = min(4, len(imgs_b))
fig, axes = plt.subplots(n_show, 5, figsize=(18, n_show * 4 + 1), facecolor='#060d14')
if n_show == 1:
    axes = axes[np.newaxis, :]

col_titles = ['Ảnh MRI gốc', 'Ground Truth', 'Prediction', 'Overlay', 'TP/FP/FN map']
for j, title in enumerate(col_titles):
    axes[0, j].set_title(title, color='#e8f4f8', fontsize=10, pad=6)

for i in range(n_show):
    img_np   = denormalize(imgs_b[i])
    gt_mask  = masks_b[i, 0].numpy()
    pred_prob= probs_b[i, 0].numpy()
    pred_bin = (pred_prob > best_thresh).astype(np.uint8)

    # Overlay: khối u predict màu xanh lá
    overlay = img_np.copy()
    overlay[pred_bin == 1] = [0, 220, 100]
    overlay_blend = cv2.addWeighted(img_np, 0.55, overlay, 0.45, 0)

    # TP/FP/FN map
    diff = np.zeros((*pred_bin.shape, 3), dtype=np.uint8)
    diff[(pred_bin == 1) & (gt_mask == 1)] = [0, 230, 100]    # TP — xanh
    diff[(pred_bin == 1) & (gt_mask == 0)] = [230, 60,  60]   # FP — đỏ
    diff[(pred_bin == 0) & (gt_mask == 1)] = [255, 200, 0]    # FN — vàng

    inter = (pred_bin * gt_mask).sum()
    d_i   = (2 * inter + 1) / (pred_bin.sum() + gt_mask.sum() + 1)

    show_list = [img_np, gt_mask * 255, pred_prob * 255, overlay_blend, diff]
    cmaps     = [None, 'gray', 'jet', None, None]

    for j, (show, cmap) in enumerate(zip(show_list, cmaps)):
        ax = axes[i, j]
        ax.set_facecolor('#000')
        ax.imshow(show.astype(np.uint8) if show.ndim == 3 else show,
                  cmap=cmap, vmin=0, vmax=255 if show.max() > 1 else 1)
        ax.axis('off')
        if j == 0:
            ax.set_ylabel(f'Dice={d_i:.3f}', color='#00c8ff', fontsize=9)

plt.suptitle(
    f'Validation Predictions — U-Net + ResNet50  |  '
    f'Threshold={best_thresh:.2f}  TTA-Dice={tta_dice_mean:.4f}',
    color='#e8f4f8', fontsize=11, y=1.005
)
plt.tight_layout()
plt.savefig('../outputs/val_predictions.png', dpi=150, bbox_inches='tight',
            facecolor='#060d14')
plt.show()
print('Saved → ../outputs/val_predictions.png')


In [ ]:
#  TRAINING HISTORY

epochs_x = range(1, len(history['train_loss']) + 1)

fig, axes = plt.subplots(1, 3, figsize=(17, 5), facecolor='#060d14')
fig.suptitle(f'Training History — U-Net + ResNet50  (Best Dice={best_dice:.4f} @ ep{best_epoch})',
             color='#e8f4f8', fontsize=12)

panels = [
    ('Loss',         [('train_loss','#00c8ff','Train Loss'), ('val_loss','#ff6b35','Val Loss')]),
    ('Dice & IoU',   [('val_dice','#00ff88',f'Val Dice (best={best_dice:.4f})'), ('val_iou','#ffd60a','Val IoU')]),
    ('LR (decoder)', [('lr','#c77dff','Decoder LR')]),
]

for ax, (title, lines) in zip(axes, panels):
    ax.set_facecolor('#0a1520')
    for sp in ax.spines.values(): sp.set_color('#1a2a3a')
    ax.tick_params(colors='#888')
    ax.set_title(title, color='#e8f4f8', fontsize=11)
    ax.set_xlabel('Epoch', color='#888')
    for key, color, label in lines:
        ax.plot(epochs_x, history[key], color=color, linewidth=2, label=label)
    if title == 'Loss':
        ax.axvline(best_epoch, color='#ffd60a', linestyle='--',
                   linewidth=1, alpha=0.7, label=f'Best ep={best_epoch}')
    ax.legend(facecolor='#060d14', labelcolor='#ccc', fontsize=9)

plt.tight_layout()
plt.savefig('../outputs/training_history.png', dpi=150,
            bbox_inches='tight', facecolor='#060d14')
plt.show()
print('Saved → ../outputs/training_history.png')


In [ ]:
#  FINAL EVALUATION — báo cáo đầy đủ

model.eval()
all_pred_flat, all_mask_flat = [], []

for imgs, masks in tqdm(val_loader, desc='Final eval'):
    imgs  = imgs.to(DEVICE)
    probs = tta_predict(model, imgs).cpu()
    pred  = (probs > best_thresh).long().numpy().ravel()
    mask  = masks.long().numpy().ravel()
    all_pred_flat.extend(pred.tolist())
    all_mask_flat.extend(mask.tolist())

all_pred = np.array(all_pred_flat)
all_mask = np.array(all_mask_flat)

inter = (all_pred * all_mask).sum()
union = all_pred.sum() + all_mask.sum()

dice_final      = (2 * inter + 1) / (union + 1)
iou_final       = (inter + 1) / (union - inter + 1)
precision_final = (inter + 1) / (all_pred.sum() + 1)
recall_final    = (inter + 1) / (all_mask.sum() + 1)
f1_final        = 2 * precision_final * recall_final / (precision_final + recall_final)

sep = '=' * 55
print(sep)
print('  FINAL EVALUATION RESULTS')
print(sep)
print(f'  Model       : {CFG["arch"].upper()} + {CFG["encoder"]}')
print(f'  Image size  : {CFG["img_size"]}×{CFG["img_size"]}')
print(f'  Threshold   : {best_thresh:.3f}  (optimal, grid search)')
print(f'  TTA         : 8× augmentations')
print(sep)
print(f'  Dice Score  : {dice_final:.4f}')
print(f'  IoU Score   : {iou_final:.4f}')
print(f'  Precision   : {precision_final:.4f}')
print(f'  Recall      : {recall_final:.4f}')
print(f'  F1 Score    : {f1_final:.4f}')
print(sep)
print(f'  Weights     : {CFG["save_dir"]}/{CFG["save_name"]}')
print(sep)

# Lưu kết quả ra JSON để dễ so sánh sau
result_dict = {
    'model'       : f'{CFG["arch"]}_{CFG["encoder"]}',
    'img_size'    : CFG['img_size'],
    'threshold'   : round(best_thresh, 3),
    'best_epoch'  : best_epoch,
    'dice'        : round(float(dice_final), 4),
    'iou'         : round(float(iou_final), 4),
    'precision'   : round(float(precision_final), 4),
    'recall'      : round(float(recall_final), 4),
    'f1'          : round(float(f1_final), 4),
}
with open('../outputs/eval_results.json', 'w') as f:
    json.dump(result_dict, f, indent=2)
print('Saved → ../outputs/eval_results.json')
